# ML-AL Master Code

In [108]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
import os
import random
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from ibug import IBUGWrapper
from sklearn.utils import resample
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions.normal import Normal
from torch.utils.data import TensorDataset, DataLoader
# ignore warnings
import warnings
warnings.filterwarnings("ignore")



class GPR:
    def __init__(self, kernel=None, alpha=1.0, random_state=42):
        self.alpha = alpha
        self.kernel = kernel if kernel else (
            C(1.0, (1e-5, 1e5)) * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e3)) +
            WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-3, 1e6))
        )
        self.model = GaussianProcessRegressor(kernel=self.kernel, n_restarts_optimizer=10, random_state=random_state)


    def update_model(self, X_train, y_train):
        self.model.fit(X_train, y_train)
        print("Selected kernel after optimization:", self.model.kernel_)

    def select_next_point(self, X_candidates):
        mean, std = self.model.predict(X_candidates, return_std=True)
        ucb = mean + self.alpha * std
        return np.argmax(ucb), ucb, mean, std

class RFR:
    def __init__(self,n_estimators=400, alpha=1.0, random_state=42):
        self.alpha = alpha
        self.model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)

    def update_model(self, X_train, y_train):
        self.model.fit(X_train, y_train)
    
    def rfPredictionIntervals(self, xVal, percentile=90):
        # initialize a list to hold the predictions from each tree
        y_preds = []
        # loop through the trees in the random forest
        for tree in self.model.estimators_:
            # get the predictions from each tree
            y_pred = tree.predict(xVal)
            # append the predictions to the list
            y_preds.append(y_pred)
        # Convert to np.array by stacking list of arrays along the column axis with each column being the prediction from a different tree
        y_preds = np.stack(y_preds, axis=1)           
        # get the quantiles for the confidence interval
        q_down = (100 - percentile) / 2.
        q_up = 100 - q_down

        # get the mean, uncertainty, lower bound, and upper bound
        y_lower = np.percentile(y_preds, q_down, axis=1)
        y_upper = np.percentile(y_preds, q_up, axis=1)  
        y_mean = self.model.predict(xVal)  
        y_uncert = y_upper - y_lower
        
        return y_mean, y_uncert


    def select_next_point(self, X_candidates):
        mean, uncertainty = self.rfPredictionIntervals(X_candidates)
        ucb = mean + self.alpha * uncertainty
        return np.argmax(ucb), ucb, mean, uncertainty

class XGB:
    def __init__(self, n_estimators=400, n_models=30, alpha = 1.0, random_state=42):
        self.alpha = alpha
        self.n_models = n_models
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.models = []

    def update_model(self, X_train, y_train):
        self.models = []
        rng = np.random.RandomState(self.random_state)
        for i in range(self.n_models):
            X_sample, y_sample = resample(X_train, y_train, random_state=rng.randint(0, 10000))
            model = XGBRegressor(
                n_estimators=self.n_estimators,
                reg_alpha=0,
                scale_pos_weight=1,
                base_score=0.5,
                random_state=rng.randint(0, 10000)
            )
            model.fit(X_sample, y_sample)
            self.models.append(model)

    def select_next_point(self, X_candidates):
        # Predict with all models
        y_preds = np.column_stack([model.predict(X_candidates) for model in self.models])

        # Compute uncertainty and mean
        y_mean = np.mean(y_preds, axis=1)
        y_std = np.std(y_preds, axis=1)
        ucb = y_mean + self.alpha * y_std
        return np.argmax(ucb), ucb, y_mean, y_std


#-------------------------------------------------------------------------------- BAYESIAN NEURAL NETWORK -------------------------------------------------------------------------------------------
class BayesianLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        # Variational parameters for weights and biases
        self.weight_mu = nn.Parameter(torch.Tensor(out_features, in_features).uniform_(-0.2, 0.2))
        self.weight_log_sigma = nn.Parameter(torch.Tensor(out_features, in_features).fill_(-5.0))

        self.bias_mu = nn.Parameter(torch.Tensor(out_features).uniform_(-0.2, 0.2))
        self.bias_log_sigma = nn.Parameter(torch.Tensor(out_features).fill_(-5.0))

        # Prior
        self.prior = Normal(0, 1)
        self.normal = Normal(0, 1)

    def forward(self, x):
        device = x.device

        weight_sigma = torch.exp(self.weight_log_sigma)
        bias_sigma = torch.exp(self.bias_log_sigma)

        # Create ε ~ N(0, 1) on the correct device
        weight_eps = torch.randn_like(self.weight_mu, device=device)
        bias_eps = torch.randn_like(self.bias_mu, device=device)

        weight = self.weight_mu + weight_sigma * weight_eps
        bias = self.bias_mu + bias_sigma * bias_eps

        return F.linear(x, weight, bias)


    def kl_loss(self):
        # Posterior: N(mu, sigma^2), Prior: N(0, 1)
        weight_sigma = torch.exp(self.weight_log_sigma)
        bias_sigma = torch.exp(self.bias_log_sigma)

        weight_kl = (torch.log(1.0 / weight_sigma) + (weight_sigma ** 2 + self.weight_mu ** 2 - 1) / 2).sum()
        bias_kl = (torch.log(1.0 / bias_sigma) + (bias_sigma ** 2 + self.bias_mu ** 2 - 1) / 2).sum()
        return weight_kl + bias_kl


class BayesianNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.layers = nn.ModuleList([
            BayesianLinear(input_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
        ])
        self.out = BayesianLinear(hidden_dim, 1)

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return self.out(x)

    def kl_loss(self):
        return sum(layer.kl_loss() for layer in self.layers) + self.out.kl_loss()


def elbo_loss(predictions, targets, kl, beta=1.0):
    mse = F.mse_loss(predictions.squeeze(), targets, reduction='mean')
    return 0.5*mse + beta * kl


def train_bnn(model, X_train, y_train, n_epochs=1000, lr=1e-3, beta=1.0, batch_size=64, device='cpu'):
    # Create DataLoader for batching
    dataset = TensorDataset(X_train, y_train)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.to(device)

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        total_kl = 0.0
        total_batches = 0

        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            preds = model(X_batch)
            kl = model.kl_loss()
            loss = elbo_loss(preds, y_batch, kl, beta)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_kl += kl.item()
            total_batches += 1

        if epoch % 100 == 0:
            avg_loss = total_loss / total_batches
            avg_kl = total_kl / total_batches
            print(f"Epoch {epoch}, Avg Loss: {avg_loss:.4f}, Avg KL: {avg_kl:.2f}")


def predict_with_uncertainty(model, X, n_samples=1000):
    model.eval()
    #with torch.no_grad():
    preds = torch.stack([model(X).detach().squeeze() for _ in range(n_samples)])

    mean = preds.mean(0).cpu().numpy()
    std = preds.std(0).cpu().numpy()
    return mean, std

#-------------------------------------------------------------------------------- END CODE BAYESIAN NEURAL NETWORK -------------------------------------------------------------------------------------------


class BNN:
    def __init__(self, input_dim,device='cpu', alpha=1.0):
        self.alpha = alpha
        self.device=device
        self.model= BayesianNN(input_dim=input_dim).to(self.device)

    def update_model(self, X_train, y_train):
        X_train_tensor=torch.from_numpy(X_train).float().to(self.device)
        y_train_tensor=torch.from_numpy(y_train).float().to(self.device)
        train_bnn(self.model,X_train_tensor, y_train_tensor, batch_size=64,device=self.device)

    def select_next_point(self, X_candidates):
        X_candidates_tensor=torch.from_numpy(X_candidates).float().to(self.device)
        mean, std = predict_with_uncertainty(self.model,X_candidates_tensor)
        ucb = mean + self.alpha * std
        return np.argmax(ucb), ucb, mean, std





def run_MLAL(X_df, y_df, model_name, target_name, dataset_name, alpha=1.0, random_seed=42):
    X = X_df.values
    y = y_df.values
    max_target = y.max()

    num_col=X.shape[1]

    random.seed(random_seed)
    np.random.seed(random_seed)

    initial_idx = random.choice(list(range(len(X))))
    X_train = X[[initial_idx]]
    y_train = np.array([y[initial_idx]])

    if model_name == "GPR":
        model = GPR(alpha=alpha, random_state=random_seed)
    elif model_name == "RFR":
        model = RFR(alpha=alpha, random_state=random_seed)
    elif model_name == "XGB":
        model = XGB(alpha=alpha, random_state=random_seed)
    elif model_name == "BNN":
        model = BNN(num_col, alpha=alpha, device='cuda' if torch.cuda.is_available() else 'cpu')
    else:
        raise ValueError("Invalid model name. Choose from 'GPR', 'RFR', 'XGB', or 'BNN'.")
    

    # Indices of selected points
    selected = [initial_idx]
    # Observed values
    observed = [y[initial_idx]]
    # Mean predictions over all candidates
    mean_predictions = []
    # Uncertainty estimates
    uncertaintites = []

    iteration_indices = [0]
    trajectory_data = []

    for i, _ in enumerate(range(len(X) - 1)):
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)

        model.update_model(X_train_scaled, y_train)
        available = list(set(range(len(X))) - set(selected))

        X_candidates_scaled = scaler.transform(X[available])


        idx, ucb_scores, mean, uncertaintity = model.select_next_point(X_candidates_scaled)
        next_idx = available[idx]
        selected.append(next_idx)
        observed.append(y[next_idx])
        X_train = np.vstack([X_train, X[[next_idx]]])
        y_train= np.append(y_train, y[next_idx])

        mean_predictions.append(mean[idx])
        uncertaintites.append(uncertaintity[idx])

        iteration_indices.append(i+1)

        trajectory_data.append({
            "Iteration": i+1,
            "Index": next_idx,
            "Observed Target Value": y[next_idx],
            "Predicted Target Value": mean[idx],
            "Uncertainty": uncertaintity[idx],
            f"Max {target_name} in Dataset": max_target,
            "Stopping Reason": f"Max {target_name} reached" if y[next_idx] >= max_target else "Continuing"
        })

        if y[next_idx] >= max_target:
            print(f"Stopping early at iteration {i+1} - Max {target_name} found.")
            break

    df_traj = pd.DataFrame(trajectory_data)
    os.makedirs(f"al_trajectory_data_all/{dataset_name}", exist_ok=True)
    file_path = f"al_trajectory_data_all/{dataset_name}/{model_name}_trajectory_{dataset_name}_alpha{alpha}_seed{random_seed}.csv"
    df_traj.to_csv(file_path, index=False)
    print(f"Saved trajectory data to {file_path}")

    # Plotting
    plt.figure(figsize=(10, 6))
    plt.plot(range(len(observed)), observed, marker='o', label=f"Observed with {model_name} (alpha={alpha})")
    plt.fill_between(iteration_indices[1:], np.array(mean_predictions) - np.array(uncertaintites),
                     np.array(mean_predictions) + np.array(uncertaintites),
                     color="gray", alpha=0.3, label=f"Uncertainty {model_name}")
    plt.plot(iteration_indices[1:], mean_predictions, linestyle='--', color="gray", label=f"Predicted Strength using {model_name}")


    plt.axhline(max_target, color='r', linestyle='--', label=f'Max {target_name}')
    plt.xlabel("Iteration")
    plt.ylabel(target_name)
    plt.title(f"{model_name} Active Learning Trajectory (Alpha = {alpha}; Seed={random_seed})")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


## matbench_steels

In [ ]:
data_grouped = pd.read_csv('matbench_steels.csv')
X_df = data_grouped.drop(columns=['composition_original','yield strength','Report', 'Report with output', 'Formatted_Parameters'], errors='ignore')
X_df = data_grouped.iloc[:, 2:16]
y_df = data_grouped['yield strength']

In [3]:
for model_name in ["GPR", "RFR", "XGB", "BNN"]:
    for alpha in [0, 0.1, 0.3, 0.5, 0.8, 1, 2, 3, 4, 5]:
        for seed in [42, 41, 40, 39, 38]:
            run_MLAL(X_df, y_df, model_name=model_name, target_name="Yield Strength", dataset_name="matbench_steels (composition)", alpha=alpha, random_seed=seed)

## P3HT

In [6]:
data_grouped = pd.read_csv("P3HT_dataset.csv")
X_df = data_grouped.iloc[:, 0:5]
#X_df = data_grouped.iloc[:, 2:16]
y_df = data_grouped['Conductivity (measured) (S/cm)']

In [ ]:
for model_name in ["GPR", "RFR", "XGB", "BNN"]:
    for alpha in [0, 0.1, 0.3, 0.5, 0.8, 1, 2, 3, 4, 5]:
        for seed in [42, 41, 40, 39, 38]:
            run_MLAL(X_df, y_df, model_name=model_name, target_name="Conductivity", dataset_name="P3HT_dataset", alpha=alpha, random_seed=seed)

## Perovskite

In [13]:
data_grouped = pd.read_csv("Perovskite_dataset.csv")
X_df = data_grouped.iloc[:, 0:3]
#X_df = data_grouped.iloc[:, 2:16]
y_df = -data_grouped['Instability index']
y_df

In [ ]:
for model_name in ["GPR", "RFR", "XGB", "BNN"]:
    for alpha in [0, 0.1, 0.3, 0.5, 0.8, 1, 2, 3, 4, 5]:
        for seed in [42, 41, 40, 39, 38]:
            run_MLAL(X_df, y_df, model_name=model_name, target_name="-Instability index", dataset_name="Perovskite_dataset", alpha=alpha, random_seed=seed)

## Membrane Dataset

In [ ]:
data_grouped = pd.read_csv('Membrane_dataset.csv')
X_df = data_grouped.drop([
    'Elastic Modulus_mean', 'Elastic Modulus_std', 'Yield Strength_mean',
    'Yield Strength_std', 'Creep Strain_mean', 'Creep Strain_std',
    'Plateau Slope_mean', 'Plateau Slope_std', 'Densification Slope_mean',
    'Densification Slope_std', 'Changepoint_mean', 'Changepoint_std', 'Date',
    'Average Standard Deviation_mean', 'Average Standard Deviation_std',
    'Coefficient of Variation_mean', 'Coefficient of Variation_std',
    'Coefficient of Variation', 'Batch', 'Sample', 'Index', 'Formatted_Parameters','Report','Report with output', 'Formatted_Parameters with output'], axis=1)
y_df = data_grouped['Elastic Modulus_mean']
X_df

In [ ]:
for model_name in ["GPR", "RFR", "XGB", "BNN"]:
    for alpha in [0, 0.1, 0.3, 0.5, 0.8, 1, 2, 3, 4, 5]:
        for seed in [42, 41, 40, 39, 38]:
            run_MLAL(X_df, y_df, model_name=model_name, target_name="Elastic Modulus", dataset_name="Membrane_dataset", alpha=alpha, random_seed=seed)

# Random Walk

In [3]:
# ============================================================
# Random-walk trajectory generator & saver (AL-style init)
# - Initial point via rng.choice(...)  ✅ matches your AL snippet
# - Remaining order via rng.shuffle(...)
# - Stop when the global best is first encountered
# - Deterministic per (dataset, seed)
# - CSV columns: Iteration, Index, Observed Target Value
# - Filenames include "seed" and "trajectory" for your parser
# ============================================================

import os
import random
from typing import Sequence, Union, Dict, Any, List, Optional
import numpy as np
import pandas as pd

def make_random_walk_traj(
    data: Union[str, pd.DataFrame],
    y_col: str,
    seed: int,
    goal: str = "max",
    index_is_one_based: bool = False,
) -> pd.DataFrame:
    """
    Build a random-walk trajectory:
      1) initial index chosen by rng.choice(valid_idx)  (AL-style)
      2) remaining indices shuffled
      3) traverse until the global best (by goal) is first hit
    Returns a DataFrame with:
      - Iteration: 0..(path_len-1)
      - Index: integer index into the data (0- or 1-based per flag)
      - Observed Target Value: y at that index
    """
    data_df = pd.read_csv(data) if isinstance(data, str) else data.copy()

    # valid indices where target is not NaN
    valid_idx = data_df[y_col].dropna().index.to_list()
    if not valid_idx:
        raise ValueError("No valid rows with target present for random walk.")

    # Deterministic RNGs
    rng = random.Random(seed)
    np.random.seed(seed)

    # Find global best position under the chosen goal
    y_vals = data_df.loc[valid_idx, y_col].to_numpy()
    if goal not in ("max", "min"):
        raise ValueError("goal must be 'max' or 'min'")
    best_pos = valid_idx[int(np.nanargmax(y_vals) if goal == "max" else np.nanargmin(y_vals))]

    # --- AL-style initial selection ---
    initial_idx = rng.choice(valid_idx)
    # Shuffle the rest
    remaining = [i for i in valid_idx if i != initial_idx]
    rng.shuffle(remaining)
    order = [initial_idx] + remaining

    # Walk until we hit the global best
    path = []
    for ridx in order:
        path.append(ridx)
        if ridx == best_pos:
            break

    # Build trajectory rows
    rows = []
    for it, ridx in enumerate(path):
        label = (ridx + 1) if index_is_one_based else ridx
        rows.append({
            "Iteration": it,
            "Index": label,
            "Observed Target Value": float(data_df.loc[ridx, y_col]),
        })

    return pd.DataFrame(rows)


def save_random_walks_for_configs(
    configs: List[Dict[str, Any]],
    seeds: Sequence[int],
    *,
    out_subdir: str = "random_walk",
    index_is_one_based: bool = False,
) -> List[str]:
    """
    For each dataset config, generate and save random-walk trajectories for each seed.

    Required keys per config:
      - folder: output folder (typically the dataset folder you already use)
      - data_csv: path to the dataset CSV
      - y_col: target column name
      - goal: "max" or "min" (optional, default "max")
      - ds_key (optional): only used for logging/file naming context

    Returns a list of saved file paths.
    """
    saved_paths: List[str] = []
    for cfg in configs:
        data_csv = cfg["data_csv"]
        y_col    = cfg["y_col"]
        goal     = cfg.get("goal", "max")
        ds_key   = cfg.get("ds_key") or os.path.splitext(os.path.basename(data_csv))[0]

        # Put outputs under <folder>/<out_subdir>/
        folder = cfg["folder"]
        out_dir = os.path.join(folder, out_subdir)
        os.makedirs(out_dir, exist_ok=True)

        for s in seeds:
            traj = make_random_walk_traj(
                data=data_csv,
                y_col=y_col,
                seed=s,
                goal=goal,
                index_is_one_based=index_is_one_based,
            )

            # Include tokens "RandomWalk", "seed", and "trajectory" for downstream parsers
            fname = f"RandomWalk_seed{s}_trajectory.csv"
            fpath = os.path.join(out_dir, fname)
            traj.to_csv(fpath, index=False)
            saved_paths.append(fpath)

    return saved_paths


# ============================
# Example usage
# (Drop in your existing configs block and run)
# ============================
if __name__ == "__main__":
    base_dir = "al_trajectory_data_all"
    configs = [
        {
            "ds_key": "matbench_steels (composition)",
            "folder": os.path.join(base_dir, "matbench_steels (composition)"),
            "data_csv": "steels_yield_report.csv",
            "y_col": "yield strength",
            "goal": "max",
        },
        {
            "ds_key": "P3HT_dataset",
            "folder": os.path.join(base_dir, "P3HT_dataset"),
            "data_csv": "P3HT_dataset_report.csv",
            "y_col": "Conductivity (measured) (S/cm)",
            "goal": "max",
        },
        {
            "ds_key": "Perovskite_dataset",
            "folder": os.path.join(base_dir, "Perovskite_dataset"),
            "data_csv": "Perovskite_dataset_report.csv",
            "y_col": "Instability index",
            "goal": "min",
        },
        {
            "ds_key": "Membrane_dataset",
            "folder": os.path.join(base_dir, "Membrane_dataset"),
            "data_csv": "all_data_parameters_2024_11_26_forAL_grouped_2025_05_27.csv",
            "y_col": "Elastic Modulus_mean",
            "goal": "max",
        },
    ]

    seeds = (38, 39, 40, 41, 42)
    paths = save_random_walks_for_configs(
        configs,
        seeds,
        out_subdir="random_walk",
        index_is_one_based=False
    )
    print("Saved random-walk trajectories:")
    for p in paths:
        print(" -", p)
